In [ ]:
import os
import sqlite3
from pathlib import Path

import pandas as pd

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.dirname(os.getcwd()))

In [ ]:
def load_tables_from_db(
    db_path: Path, tables: list[str] = ["Recordings", "Speakers"]
) -> tuple[pd.DataFrame, ...]:
    """Load specified tables from a SQLite database and return them as a tuple of data frames.

    Args:
        db_path: The path to the SQLite database file.
        tables: A list of table names to load. Defaults to ["Recordings", "Speakers"].

    Returns:
        A tuple containing pandas DataFrames for the specified tables.
    """
    conn = sqlite3.connect(db_path)
    dataframes = []
    for table in tables:
        query = f"SELECT * FROM {table}"
        dataframe = pd.read_sql_query(query, conn)
        dataframes.append(dataframe)
    conn.close()
    return tuple(dataframes)

In [ ]:
# Define the path to the public database
path_database_public = Path("new.db")

In [ ]:
df_recordings, df_conversations = load_tables_from_db(
    path_database_public, ["Recordings", "Conversations"]
)

In [ ]:
df_combined = pd.concat([df_recordings, df_conversations], ignore_index=True)
df_combined["id"] = df_combined["id_recording"].combine_first(
    df_combined["id_conversation"]
)

In [ ]:
df_combined.columns

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
from pydub import AudioSegment
from tqdm.auto import tqdm


def attach_audio_paths(df, data_root, extensions=None):
    """Adds 'audio_path' column by locating audio files whose stem matches df['id'].

    Args:
        df (pd.DataFrame): DataFrame containing an 'id' column.
        data_root (str or Path): Root directory to search for audio files.
        extensions (set of str, optional): Set of allowed file extensions.

    Returns:
        pd.DataFrame: DataFrame with added 'audio_path' column.
    """
    df = df.copy()
    df["id"] = df["id"].astype(str)

    data_root = Path(data_root)

    if extensions:
        extensions = {ext.lower() for ext in extensions}

    file_map = {}

    all_files = list(data_root.rglob("*"))

    for path in tqdm(all_files, desc="Indexing audio files"):
        if path.is_file():
            if extensions and path.suffix.lower() not in extensions:
                continue
            file_map.setdefault(path.stem, str(path))

    df["audio_path"] = df["id"].map(file_map)

    return df


def get_audio_duration(path):
    """Returns duration in seconds.

    Args:
        path (str or Path): Path to the audio file.

    Returns:
        float: Duration of the audio file in seconds, or np.nan if it cannot be determined.
    """
    path = Path(path)
    try:
        # Try soundfile first (fast, header-only)
        info = sf.info(path)
        return info.frames / info.samplerate
    except RuntimeError:
        # Fallback for unsupported formats like m4a
        try:
            audio = AudioSegment.from_file(path)
            return len(audio) / 1000  # milliseconds -> seconds
        except Exception:
            return np.nan


def attach_audio_durations(df, path_column="audio_path"):
    """Adds 'duration_sec' column using header metadata only.

    Args:
        df (pd.DataFrame): DataFrame containing audio file paths.
        path_column (str): Column name containing audio file paths.

    Returns:
        pd.DataFrame: DataFrame with added 'duration_sec' column.
    """
    df = df.copy()

    tqdm.pandas(desc="Computing durations")

    df["duration"] = df[path_column].progress_apply(
        lambda x: get_audio_duration(x) if pd.notna(x) else np.nan
    )

    return df

In [ ]:
path_audio = Path("/mnt/hdd/data/raw/")

df = attach_audio_paths(df_combined, path_audio)
df = attach_audio_durations(df)

In [ ]:
df

In [ ]:
# Keep only required columns
df_duration = df[["id", "duration_sec"]].copy()

# Save to SQLite
with sqlite3.connect(path_database_public) as conn:
    df_duration.to_sql(
        "Durations",
        conn,
        if_exists="replace",  # change to "append" if needed
        index=False,
    )

In [ ]:
df_dur = load_tables_from_db(path_database_public, ["Durations"])

In [ ]:
df_dur